In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report
import os

In [2]:
import numpy as np
import pandas as pd
import os

base_path = "/content/drive/MyDrive/Colab Notebooks/processed_features/slider"

id_folders = ['id_00', 'id_02', 'id_04', 'id_06']

datasets = {}
for id_folder in id_folders:
    records = []

    for label in ['normal', 'abnormal']:
        folder_path = os.path.join(base_path, id_folder, label)

        for file in sorted(os.listdir(folder_path)):
            if file.endswith(".npy"):
                file_path = os.path.join(folder_path, file)
                data = np.load(file_path, allow_pickle=True).flatten()

                row = {f'col_{i}': val for i, val in enumerate(data)}
                row['label'] = int(id_folder[-1])/2

                records.append(row)

    datasets[id_folder] = pd.DataFrame(records)
    print(f"{id_folder} -> shape: {datasets[id_folder].shape}")



id_00 -> shape: (1424, 9)
id_02 -> shape: (1335, 9)
id_04 -> shape: (712, 9)
id_06 -> shape: (623, 9)


In [3]:
id_00 = datasets['id_00']
id_02 = datasets['id_02']
id_04 = datasets['id_04']
id_06 = datasets['id_06']

In [4]:
data = pd.concat([id_00,id_04,id_02,id_06], ignore_index=True)

In [5]:
data.info()
data.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4094 entries, 0 to 4093
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   col_0   4094 non-null   float32
 1   col_1   4094 non-null   float32
 2   col_2   4094 non-null   float32
 3   col_3   4094 non-null   float32
 4   col_4   4094 non-null   float32
 5   col_5   4094 non-null   float32
 6   col_6   4094 non-null   float32
 7   col_7   4094 non-null   float32
 8   label   4094 non-null   float64
dtypes: float32(8), float64(1)
memory usage: 160.1 KB


,col_0,col_1,col_2,col_3,col_4,col_5,col_6,col_7,label
0,0.117114,0.000159,3.794655e-08,80.060928,1209.733276,2283.146973,-877.769592,110.741287,0.0
1,0.182670,0.000232,8.984456e-08,0.000000,1804.905762,3357.178467,-824.700684,83.827568,0.0
2,0.153777,0.000183,5.154827e-08,514.261963,1723.986450,3504.068604,-875.503479,88.477531,0.0
3,0.157713,0.000200,6.430528e-08,830.609375,1732.564697,3573.981689,-852.548035,77.285034,0.0
4,0.191838,0.000188,5.285149e-08,735.726807,1954.067261,3734.749512,-855.136597,72.530525,0.0


In [6]:
import torch.nn as nn
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

class Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.input     = nn.Linear(8, 64)
        self.bn1       = nn.BatchNorm1d(64)
        self.dropout1  = nn.Dropout(0.3)

        self.fc1       = nn.Linear(64, 128)
        self.bn2       = nn.BatchNorm1d(128)
        self.dropout2  = nn.Dropout(0.3)

        self.fc2       = nn.Linear(128, 64)
        self.bn3       = nn.BatchNorm1d(64)
        self.dropout3  = nn.Dropout(0.2)

        self.output    = nn.Linear(64, 4)

        self.optimizer = torch.optim.Adam(self.parameters(), lr=0.001, weight_decay=1e-4)
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(self.optimizer, patience=20, factor=0.5)
        self.loss_fun  = nn.CrossEntropyLoss()

    def forward(self, x):
        x = F.relu(self.bn1(self.input(x)))
        x = self.dropout1(x)
        x = F.relu(self.bn2(self.fc1(x)))
        x = self.dropout2(x)
        x = F.relu(self.bn3(self.fc2(x)))
        x = self.dropout3(x)
        return self.output(x)

    def train_model(self, X, y):

        numepochs = 2500
        X_tensor = torch.tensor(X.values, dtype=torch.float32)
        y_tensor = torch.tensor(y.values, dtype=torch.long)

        train_data_tensor, test_data_tensor, train_labels_tensor, test_labels_tensor = train_test_split(
            X_tensor, y_tensor, test_size=0.1, random_state=42, stratify=y_tensor
        )

        train_dataset = TensorDataset(train_data_tensor, train_labels_tensor)
        test_dataset  = TensorDataset(test_data_tensor,  test_labels_tensor)

        batchsize    = 64
        train_loader = DataLoader(train_dataset, batch_size=batchsize, shuffle=True, drop_last=True)
        test_loader  = DataLoader(test_dataset,  batch_size=test_dataset.tensors[0].shape[0])

        losses   = torch.zeros(numepochs)
        trainAcc = []
        testAcc  = []

        best_test_acc = 0
        best_weights  = None

        for epochi in range(numepochs):
            self.train()
            batchAcc  = []
            batchLoss = []

            for X_batch, y_batch in train_loader:
                yHat = self(X_batch)
                loss = self.loss_fun(yHat, y_batch)

                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()

                batchLoss.append(loss.item())
                matches        = torch.argmax(yHat, axis=1) == y_batch
                accuracyPct    = 100 * torch.mean(matches.float())
                batchAcc.append(accuracyPct)

            trainAcc.append(np.mean(batchAcc))
            losses[epochi] = np.mean(batchLoss)


            self.eval()
            X_test, y_test = next(iter(test_loader))
            with torch.no_grad():
                yHat = self(X_test)

            acc = 100 * torch.mean((torch.argmax(yHat, axis=1) == y_test).float())
            testAcc.append(acc)

            self.scheduler.step(losses[epochi])
            if acc > best_test_acc:
                best_test_acc = acc
                best_weights  = {k: v.clone() for k, v in self.state_dict().items()}

            if (epochi + 1) % 100 == 0:
                print(f"Epoch {epochi+1}/{numepochs} | Loss: {losses[epochi]:.4f} | Train: {trainAcc[-1]:.2f}% | Test: {acc:.2f}%")

        self.load_state_dict(best_weights)
        print(f"\nBest Test Accuracy: {best_test_acc:.2f}%")

        return trainAcc, testAcc, losses

In [7]:
X = data.drop('label', axis=1)
y = data['label']

model = Model()

In [8]:
model.train_model(X,y)

Epoch 100/2500 | Loss: 0.5266 | Train: 76.21% | Test: 81.71%
Epoch 200/2500 | Loss: 0.4873 | Train: 78.12% | Test: 81.71%
Epoch 300/2500 | Loss: 0.4680 | Train: 79.28% | Test: 83.17%
Epoch 400/2500 | Loss: 0.4659 | Train: 80.40% | Test: 83.17%
Epoch 500/2500 | Loss: 0.4654 | Train: 79.33% | Test: 83.41%
Epoch 600/2500 | Loss: 0.4693 | Train: 79.30% | Test: 83.66%
Epoch 700/2500 | Loss: 0.4657 | Train: 79.63% | Test: 83.90%
Epoch 800/2500 | Loss: 0.4552 | Train: 80.54% | Test: 83.41%
Epoch 900/2500 | Loss: 0.4711 | Train: 79.33% | Test: 83.41%
Epoch 1000/2500 | Loss: 0.4655 | Train: 79.80% | Test: 83.66%
Epoch 1100/2500 | Loss: 0.4831 | Train: 78.76% | Test: 83.66%
Epoch 1200/2500 | Loss: 0.4740 | Train: 79.03% | Test: 82.68%
Epoch 1300/2500 | Loss: 0.4695 | Train: 78.84% | Test: 82.93%
Epoch 1400/2500 | Loss: 0.4781 | Train: 78.45% | Test: 83.66%
Epoch 1500/2500 | Loss: 0.4794 | Train: 79.30% | Test: 83.17%
Epoch 1600/2500 | Loss: 0.4583 | Train: 79.66% | Test: 83.41%
Epoch 1700/2500 |

([np.float32(48.053726),
  np.float32(55.53728),
  np.float32(60.663376),
  np.float32(61.67763),
  np.float32(62.664474),
  np.float32(65.48794),
  np.float32(67.43421),
  np.float32(67.242325),
  np.float32(67.65351),
  np.float32(69.81908),
  np.float32(69.791664),
  np.float32(70.91557),
  np.float32(70.038376),
  np.float32(70.5318),
  np.float32(69.76425),
  np.float32(71.820175),
  np.float32(70.9704),
  np.float32(71.54605),
  np.float32(71.18969),
  np.float32(72.36842),
  np.float32(72.58772),
  np.float32(71.875),
  np.float32(72.9989),
  np.float32(72.34101),
  np.float32(72.20395),
  np.float32(72.3136),
  np.float32(73.13596),
  np.float32(72.7796),
  np.float32(72.72478),
  np.float32(73.41009),
  np.float32(72.97149),
  np.float32(73.46491),
  np.float32(73.382675),
  np.float32(72.7796),
  np.float32(74.89035),
  np.float32(74.50658),
  np.float32(73.958336),
  np.float32(73.4375),
  np.float32(73.79386),
  np.float32(74.45175),
  np.float32(73.711624),
  np.float32(73

In [10]:
from joblib import dump


In [11]:
dump(model,'slider_model.joblib')

['slider_model.joblib']